# Todo List — baseline tải trên Kaggle CPU

Chọn **Accelerator = None**. Bật Internet nếu clone GitHub/cài từ PyPI. Notebook gọi
`kaggle/benchmark.py` hiện có, không thay thế script. Chạy các cell theo thứ tự.

- Mặc định: smoke `1 × 25`; baseline concurrency `1, 4, 8`, 100 vòng/user, 3 lần/mức.
- Một Uvicorn process; database SQLite mới **mỗi lần chạy**, cùng cấu hình warm-up/timeout.
- Mỗi batch có thư mục riêng; giữ cả lần lỗi, không ghi đè kết quả cũ.
- Server/client cùng máy: kết quả bao gồm cạnh tranh tài nguyên giữa hai tiến trình.
- Notebook chưa chứa kết quả Kaggle. Chỉ coi là bằng chứng Kaggle sau khi bạn thực thi tại đó.

In [ ]:
import csv
import hashlib
import json
import os
from pathlib import Path
import platform
import secrets
import shutil
import signal
import socket
import subprocess
import sys
import tempfile
import time
import urllib.error
import urllib.request
import uuid
from datetime import datetime, timezone

# Nếu đã thêm source dưới dạng Dataset, đặt đúng thư mục chứa app/ và requirements.txt.
# None: tự clone repository vào thư mục làm việc mới.
SOURCE_DIR = None
REPO_URL = "https://github.com/kiennkt05/Todo-List.git"
# Khuyến nghị điền SHA commit đã push chứa benchmark đợt 3. Để trống: dùng HEAD và ghi SHA thực tế.
SOURCE_COMMIT = ""
WORK_ROOT = Path("/kaggle/working")
BASE_PYTHON = sys.executable  # Có thể đổi thành đường dẫn Python 3.12 đã cài.
INSTALL_DEPENDENCIES = True

CONCURRENCIES = [1, 4, 8]
ITERATIONS = 100
REPEATS = 3
SMOKE_ITERATIONS = 25
WARMUP_ITERATIONS = 5
HTTP_TIMEOUT = 30
STARTUP_TIMEOUT = 30
RUN_TIMEOUT = 1800  # Giới hạn toàn subprocess benchmark; khác timeout HTTP.
UVICORN_WORKERS = 1
DATABASE_POLICY = "fresh_database_per_run"

BATCH_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
BATCH_DIR = WORK_ROOT / "benchmark-results" / BATCH_ID
BATCH_DIR.mkdir(parents=True, exist_ok=False)
print("Batch:", BATCH_DIR)

## Chuẩn bị source và môi trường

Mã clone và virtualenv đặt ngoài thư mục bằng chứng. Không checkout đè lên SOURCE_DIR do bạn cung cấp;
nếu điền SOURCE_COMMIT, notebook xác nhận source khớp commit đó. Source tải ZIP không có Git vẫn dùng được
khi SOURCE_COMMIT trống; SHA và dirty sẽ là null, nhưng notebook lưu hash các file mã nguồn.

Notebook dùng phiên bản BASE_PYTHON thực tế, **không mặc định khẳng định là Python 3.12**.
Muốn so sánh với local 3.12, cấu hình interpreter phù hợp trước khi chạy.
Nếu offline, cần chuẩn bị source và môi trường có dependencies, đặt INSTALL_DEPENDENCIES=False.

In [ ]:
if SOURCE_DIR is None:
    PROJECT_DIR = WORK_ROOT / ("todo-source-" + BATCH_ID)
    subprocess.run(["git", "clone", "--quiet", REPO_URL, str(PROJECT_DIR)], check=True)
    if SOURCE_COMMIT:
        subprocess.run(["git", "checkout", "--detach", SOURCE_COMMIT], cwd=PROJECT_DIR, check=True)
else:
    PROJECT_DIR = Path(SOURCE_DIR).resolve()
for required in ("requirements.txt", "app/main.py", "kaggle/benchmark.py"):
    if not (PROJECT_DIR / required).is_file():
        raise FileNotFoundError(f"Source thiếu {required}: {PROJECT_DIR}")

def git_output(*args):
    try:
        return subprocess.run(["git", *args], cwd=PROJECT_DIR, check=True,
                              capture_output=True, text=True, timeout=10).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        return None

commit = git_output("rev-parse", "HEAD")
status = git_output("status", "--porcelain", "--untracked-files=normal")
if SOURCE_COMMIT and (commit is None or commit != git_output("rev-parse", SOURCE_COMMIT + "^{commit}")):
    raise RuntimeError("Source không khớp SOURCE_COMMIT; dừng trước khi đo")

if INSTALL_DEPENDENCIES:
    ENV_DIR = Path(tempfile.mkdtemp(prefix="todo-benchmark-env-"))
    subprocess.run([BASE_PYTHON, "-m", "venv", str(ENV_DIR)], check=True)
    RUN_PYTHON = str(ENV_DIR / "bin" / "python")
    with (BATCH_DIR / "install.log").open("w") as log:
        subprocess.run([RUN_PYTHON, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt"),
                        "matplotlib"], stdout=log, stderr=subprocess.STDOUT, check=True)
else:
    RUN_PYTHON = BASE_PYTHON
subprocess.run([RUN_PYTHON, "-c", "import fastapi, sqlalchemy, jwt, uvicorn, matplotlib"], check=True)
help_output = subprocess.run([RUN_PYTHON, str(PROJECT_DIR / "kaggle/benchmark.py"), "--help"],
                             capture_output=True, text=True, check=True).stdout
for flag in ("--warmup-iterations", "--timeout", "--output-json"):
    if flag not in help_output:
        raise RuntimeError(f"Source benchmark chưa hỗ trợ {flag}; hãy push/chọn commit đợt 3")
freeze = subprocess.run([RUN_PYTHON, "-m", "pip", "freeze"], capture_output=True, text=True, check=True).stdout
(BATCH_DIR / "dependencies.txt").write_text(freeze, encoding="utf-8")
python_version = subprocess.run([RUN_PYTHON, "--version"], capture_output=True, text=True, check=True).stdout.strip()
source_files = [PROJECT_DIR / "requirements.txt", PROJECT_DIR / "kaggle/benchmark.py", *sorted((PROJECT_DIR / "app").rglob("*.py"))]
source_hashes = {str(p.relative_to(PROJECT_DIR)): hashlib.sha256(p.read_bytes()).hexdigest() for p in source_files}
environment = {
    "batch_id": BATCH_ID, "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": commit, "git_dirty": None if status is None else bool(status),
    "source_sha256": source_hashes, "python_version": python_version,
    "platform": platform.platform(), "cpu_count": os.cpu_count(),
    "cpu_affinity_count": len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else None,
    "uvicorn_workers": UVICORN_WORKERS, "database_policy": DATABASE_POLICY,
    "server_and_client_same_machine": True,
    "concurrencies": CONCURRENCIES, "iterations": ITERATIONS, "repeats": REPEATS,
    "smoke_iterations": SMOKE_ITERATIONS, "warmup_iterations": WARMUP_ITERATIONS,
    "http_timeout_seconds": HTTP_TIMEOUT, "startup_timeout_seconds": STARTUP_TIMEOUT,
    "run_timeout_seconds": RUN_TIMEOUT,
}
if Path("/proc/meminfo").exists():
    environment["mem_total_kib"] = int(next(line.split()[1] for line in Path("/proc/meminfo").read_text().splitlines() if line.startswith("MemTotal:")))
if Path("/proc/cpuinfo").exists():
    environment["cpu_model"] = next((line.split(":", 1)[1].strip() for line in Path("/proc/cpuinfo").read_text().splitlines() if line.startswith("model name")), None)
(BATCH_DIR / "environment.json").write_text(json.dumps(environment, indent=2), encoding="utf-8")
print(python_version, "commit:", commit, "dirty:", environment["git_dirty"])
print("Đã lưu metadata; CPU/RAM hệ thống không nhất thiết bằng quota notebook.")

## Hàm quản lý một lần chạy

Mỗi lần sinh secret mới trong bộ nhớ, khởi động server với SQLite trong thư mục tạm,
chờ `/health` có deadline, gọi benchmark và luôn dừng tiến trình trong `finally`.
Không in/lưu secret. Giữ JSON và log, không đưa database/tài khoản vào gói bằng chứng.

Timeout tổng hoặc Ctrl+C: gửi SIGINT cho benchmark để nó cố lưu báo cáo; sau thời gian grace mới terminate/kill.
Nếu thiếu JSON, metadata điều phối vẫn ghi rõ lỗi. Mọi lần chạy có run directory riêng, cấm ghi đè.

In [ ]:
def write_json(path, value):
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\n", encoding="utf-8")

def stop_process(process, graceful=False):
    if process is None or process.poll() is not None:
        return
    if graceful:
        process.send_signal(signal.SIGINT)
        try:
            process.wait(timeout=3 * HTTP_TIMEOUT + 15)
            return
        except subprocess.TimeoutExpired:
            pass
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=10)

def wait_until_healthy(server, base_url):
    deadline = time.monotonic() + STARTUP_TIMEOUT
    last_error = "chưa có response"
    while time.monotonic() < deadline:
        if server.poll() is not None:
            raise RuntimeError(f"Uvicorn thoát sớm, exit code {server.returncode}; xem server.log")
        try:
            timeout = max(0.01, min(2, deadline - time.monotonic()))
            with urllib.request.urlopen(base_url + "/health", timeout=timeout) as response:
                if response.status == 200 and json.loads(response.read()) == {"status": "ok"}:
                    return
                last_error = "health status/body không đúng"
        except (OSError, ValueError) as exc:
            last_error = str(exc)
        time.sleep(min(0.1, max(0, deadline - time.monotonic())))
    raise TimeoutError(f"Không sẵn sàng sau {STARTUP_TIMEOUT}s: {last_error}")

def run_one(label, concurrency, iterations, kind="baseline", repeat=None):
    run_dir = BATCH_DIR / "runs" / label
    run_dir.mkdir(parents=True, exist_ok=False)
    report_path = BATCH_DIR / (label + ".json")
    if report_path.exists():
        raise FileExistsError(report_path)
    record = {
        "label": label, "kind": kind, "repeat": repeat, "concurrency": concurrency,
        "iterations": iterations, "orchestration_status": "starting",
        "benchmark_exit_code": None, "report_status": None, "error": None,
        "started_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    server = benchmark = None
    interrupted = False
    with tempfile.TemporaryDirectory(prefix="todo-benchmark-db-") as db_dir:
        with (run_dir / "server.log").open("w") as server_log, (run_dir / "benchmark.log").open("w") as benchmark_log:
            try:
                with socket.socket() as listener:
                    listener.bind(("127.0.0.1", 0))
                    port = listener.getsockname()[1]
                base_url = f"http://127.0.0.1:{port}"
                env = {**os.environ, "TOKEN_SECRET": secrets.token_urlsafe(48),
                       "DATABASE_URL": "sqlite:///" + str(Path(db_dir) / "todo.db")}
                server = subprocess.Popen(
                    [RUN_PYTHON, "-m", "uvicorn", "app.main:app", "--host", "127.0.0.1",
                     "--port", str(port), "--workers", str(UVICORN_WORKERS)],
                    cwd=PROJECT_DIR, env=env, stdout=server_log, stderr=subprocess.STDOUT,
                )
                wait_until_healthy(server, base_url)
                record["orchestration_status"] = "running"
                command = [RUN_PYTHON, "kaggle/benchmark.py", "--base-url", base_url,
                           "--concurrency", str(concurrency), "--iterations", str(iterations),
                           "--warmup-iterations", str(WARMUP_ITERATIONS), "--timeout", str(HTTP_TIMEOUT),
                           "--output-json", str(report_path)]
                benchmark = subprocess.Popen(command, cwd=PROJECT_DIR, stdout=benchmark_log, stderr=subprocess.STDOUT)
                record["benchmark_exit_code"] = benchmark.wait(timeout=RUN_TIMEOUT)
                record["orchestration_status"] = "finished"
            except KeyboardInterrupt:
                interrupted = True
                record.update(orchestration_status="interrupted", error="Người chạy ngắt notebook")
            except subprocess.TimeoutExpired:
                record.update(orchestration_status="timed_out", error=f"Benchmark vượt RUN_TIMEOUT={RUN_TIMEOUT}s")
            except Exception as exc:
                record.update(orchestration_status="failed", error=f"{type(exc).__name__}: {exc}")
            finally:
                try:
                    stop_process(benchmark, graceful=True)
                finally:
                    stop_process(server)
                if benchmark is not None:
                    record["benchmark_exit_code"] = benchmark.returncode
                record["server_exit_code"] = server.returncode if server is not None else None
                record["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
                try:
                    report = json.loads(report_path.read_text())
                    record["report_status"] = report["run_status"]
                    if record["orchestration_status"] == "finished":
                        good = (record["benchmark_exit_code"] == 0 and report["run_status"] == "completed"
                                and report["completed_cycles"] == concurrency * iterations
                                and report["attempted_requests"] == 3 * concurrency * iterations
                                and report["failed_requests"] == 0)
                        record["orchestration_status"] = "completed" if good else "benchmark_failed"
                except (OSError, ValueError, KeyError, TypeError) as exc:
                    record["report_error"] = f"{type(exc).__name__}: {exc}"
                    if record["orchestration_status"] == "finished":
                        record["orchestration_status"] = "invalid_report"
                write_json(run_dir / "orchestration.json", record)
    print(label, record["orchestration_status"], "report:", record["report_status"])
    if interrupted:
        raise KeyboardInterrupt("Đã dừng tiến trình và lưu metadata điều phối")
    return record

## Smoke test — phải thành công trước baseline

Mặc định có 75 request đo chính thức và 15 request warm-up. Nếu thất bại, mở
`runs/smoke/server.log`, `benchmark.log`, `orchestration.json`, `smoke.json` (nếu có).
Sửa nguyên nhân rồi bắt đầu batch mới bằng cell cấu hình; không ghi đè lần lỗi.

In [ ]:
smoke = run_one("smoke", 1, SMOKE_ITERATIONS, kind="smoke")
if smoke["orchestration_status"] != "completed":
    raise RuntimeError("Smoke thất bại: dừng baseline, xem log trong " + str(BATCH_DIR / "runs/smoke"))

## Chạy baseline — 9 lần, tuần tự

Mỗi lượt lặp chạy c1 → c4 → c8. Chỉ có các worker **bên trong** một benchmark chạy đồng thời.
Database và server mới mỗi lần. Lỗi một lần vẫn được giữ và các lần tiếp theo tiếp tục;
Ctrl+C dừng ma trận. Cell tổng hợp vẫn có thể chạy sau đó để báo những lần chưa chạy.

In [ ]:
if smoke["orchestration_status"] != "completed":
    raise RuntimeError("Cần smoke test thành công")
plan = [{"label": f"c{c}-r{r}", "concurrency": c, "iterations": ITERATIONS, "repeat": r}
        for r in range(1, REPEATS + 1) for c in CONCURRENCIES]
write_json(BATCH_DIR / "plan.json", plan)
for entry in plan:
    run_one(**entry)

## Tổng hợp CSV — không bỏ mất lần lỗi/thiếu file

Đọc đúng plan của batch, loại smoke và dữ liệu warm-up. `run_status` đến từ benchmark;
`orchestration_status` phản ánh startup, timeout, thiếu/hỏng JSON và lần chưa chạy.
Ô số liệu thiếu để trống, không đổi thành 0. `comparable` chỉ true khi run hoàn tất và cấu hình khớp batch.

In [ ]:
def summarize_batch(batch_dir):
    plan = json.loads((batch_dir / "plan.json").read_text())
    env = json.loads((batch_dir / "environment.json").read_text())
    rows = []
    for entry in plan:
        row = {"run_label": entry["label"], "repeat": entry["repeat"],
               "concurrency": entry["concurrency"], "iterations": entry["iterations"],
               "run_status": "missing", "orchestration_status": "not_run", "comparable": False, "note": ""}
        try:
            orchestration = json.loads((batch_dir / "runs" / entry["label"] / "orchestration.json").read_text())
            row["orchestration_status"] = orchestration["orchestration_status"]
            row["benchmark_exit_code"] = orchestration["benchmark_exit_code"]
            row["note"] = orchestration.get("error") or ""
        except (OSError, ValueError, KeyError, TypeError) as exc:
            row["note"] = f"Orchestration metadata: {type(exc).__name__}"
        try:
            report = json.loads((batch_dir / (entry["label"] + ".json")).read_text())
            for key in ("run_id", "run_status", "completed_cycles", "attempted_requests", "failed_requests",
                        "success_throughput", "error_rate", "git_commit", "git_dirty", "warmup_iterations", "timeout_seconds"):
                row[key] = report[key]
            for method in ("POST", "GET", "DELETE"):
                row[method.lower() + "_p95_ms"] = report["methods"][method]["p95_ms"]
            matching = (report["concurrency"] == entry["concurrency"] and report["iterations"] == entry["iterations"]
                        and report["warmup_iterations"] == env["warmup_iterations"]
                        and report["timeout_seconds"] == env["http_timeout_seconds"]
                        and report["git_commit"] == env["git_commit"])
            row["comparable"] = (matching and row["orchestration_status"] == "completed"
                                 and row["run_status"] == "completed")
            if not matching:
                row["note"] += " Configuration mismatch"
        except (OSError, ValueError, KeyError, TypeError) as exc:
            row["run_status"] = "missing" if isinstance(exc, FileNotFoundError) else "invalid_report"
            row["note"] += f" Report: {type(exc).__name__}"
        rows.append(row)
    fields = ["run_label", "repeat", "run_id", "run_status", "orchestration_status", "benchmark_exit_code",
              "comparable", "concurrency", "iterations", "completed_cycles", "attempted_requests", "failed_requests",
              "success_throughput", "error_rate", "post_p95_ms", "get_p95_ms", "delete_p95_ms",
              "git_commit", "git_dirty", "warmup_iterations", "timeout_seconds", "note"]
    with (batch_dir / "runs.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.DictWriter(file, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)
    return rows

rows = summarize_batch(BATCH_DIR)
for row in rows:
    print(row["run_label"], row["run_status"], "comparable:", row["comparable"], row["note"])
print("CSV:", BATCH_DIR / "runs.csv")

## Biểu đồ — giữ điểm riêng của từng lần lặp

Hai biểu đồ hiệu năng chỉ dùng run `comparable=True`; biểu đồ lỗi giữ các lần còn lại và đánh dấu
status. Dữ liệu thiếu là N/A, không phải 0%. Không tính p95 gộp bằng trung bình p95.
Matplotlib chạy trong cùng virtualenv đã chuẩn bị, không cần thay kernel notebook.

In [ ]:
PLOT_SCRIPT = r"""
import csv
from pathlib import Path
import sys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

batch = Path(sys.argv[1])
charts = batch / "charts"
charts.mkdir(exist_ok=True)
with (batch / "runs.csv").open(newline="", encoding="utf-8") as file:
    rows = list(csv.DictReader(file))
good = [r for r in rows if r["comparable"] == "True"]

def points(ax, key, ylabel):
    for repeat in sorted({r["repeat"] for r in good}, key=int):
        group = sorted([r for r in good if r["repeat"] == repeat and r[key] != ""], key=lambda r: int(r["concurrency"]))
        ax.scatter([int(r["concurrency"]) for r in group], [float(r[key]) for r in group], label="Repeat " + repeat)
    ax.set_xlabel("Concurrency")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    if good:
        ax.legend()
    else:
        ax.text(0.5, 0.5, "No comparable completed runs", ha="center", transform=ax.transAxes)

fig, ax = plt.subplots(figsize=(7, 4))
points(ax, "success_throughput", "Successful requests / second")
ax.set_title("Throughput: each point is one run")
fig.tight_layout()
fig.savefig(charts / "throughput.png", dpi=160)
plt.close(fig)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, method in zip(axes, ("post", "get", "delete")):
    points(ax, method + "_p95_ms", "p95 latency (ms)")
    ax.set_title(method.upper() + " successful requests")
fig.tight_layout()
fig.savefig(charts / "latency-p95.png", dpi=160)
plt.close(fig)
fig, ax = plt.subplots(figsize=(max(8, len(rows)), 4))
for index, row in enumerate(rows):
    if row["error_rate"] != "":
        value = float(row["error_rate"]) * 100
        ax.bar(index, value, color="steelblue" if row["comparable"] == "True" else "orange")
        ax.scatter([index], [value], color="black", s=12)
        label = f"{value:.2f}%"
    else:
        value, label = 0, "N/A"
    if row["comparable"] != "True":
        label += "\n" + row["orchestration_status"]
    ax.annotate(label, (index, value), xytext=(0, 5), textcoords="offset points", ha="center", fontsize=8)
ax.set_xticks(range(len(rows)), [r["run_label"] for r in rows], rotation=45, ha="right")
ax.set_ylim(0, 115)
ax.set_ylabel("Failed / recorded requests (%)")
ax.set_title("Error rate per run; N/A means no measurement")
fig.tight_layout()
fig.savefig(charts / "error-rate.png", dpi=160)
plt.close(fig)
"""
subprocess.run([RUN_PYTHON, "-c", PLOT_SCRIPT, str(BATCH_DIR)], check=True)
from IPython.display import Image, display
for name in ("throughput.png", "latency-p95.png", "error-rate.png"):
    display(Image(filename=str(BATCH_DIR / "charts" / name)))

## Lưu bằng chứng và nhận xét

Tải ZIP bên dưới và lưu notebook đã thực thi (kèm outputs). File ZIP chứa metadata, dependencies,
plan, JSON, log từng run, CSV và PNG; không chứa virtualenv/database. Không bỏ các lần lỗi khỏi bằng chứng.
Nếu chỉ chạy local để kiểm tra notebook, ghi rõ đây là local smoke, không gọi là kết quả Kaggle.

Điền nhận xét sau khi đo: throughput thay đổi theo tải, p95 của từng method, độ dao động giữa lần lặp,
lỗi/timeout gặp được và cấu hình cần đo lại. Chưa kết luận SQLite là nút thắt chỉ từ latency.

In [ ]:
archive = shutil.make_archive(str(BATCH_DIR), "zip", root_dir=BATCH_DIR)
print("Đã lưu:", archive)
from IPython.display import FileLink, display
display(FileLink(archive))